# createImageAvailabilityGraphic_inlined
Standalone copy with in-lined helper code and secrets-based Planet API loading.


## 0. Imports


In [ ]:
import datetime, json, tomllib
from datetime import timedelta
from pathlib import Path

import ee
import geopandas as gpd
import matplotlib as mpl
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from matplotlib.ticker import MultipleLocator
from requests.auth import HTTPBasicAuth
from shapely.geometry import shape


### Import and authenticate GEE


In [ ]:
def initialize_earth_engine(credential_fp):
    """Initialize Earth Engine with a service-account credential file."""
    credential_fp = Path(credential_fp)
    assert credential_fp.exists(), f"Missing Earth Engine credential file: {credential_fp}"
    credential_d = json.loads(credential_fp.read_text())
    credentials = ee.ServiceAccountCredentials(credential_d["client_email"], str(credential_fp))
    ee.Initialize(credentials=credentials, project=credential_d["project_id"])


ee_credential_fp = Path("/home/cefect/.config/USFloods_inference/ee-bryantseth.json")
initialize_earth_engine(ee_credential_fp)


### Load Planet API key from local secrets


In [ ]:
def load_secret(secret_fp, section, key):
    """Load a single secret value from the project TOML secrets file."""
    secret_fp = Path(secret_fp)
    assert secret_fp.exists(), f"Missing secrets file: {secret_fp}"
    secrets_d = tomllib.loads(secret_fp.read_text())
    value = secrets_d[section][key]
    assert value, f"Empty secret for {section}.{key}"
    return value


secrets_fp = Path('/home/cefect/.config/USFloods_inference/secrets.toml')
planetKey = load_secret(secrets_fp, 'env', 'PL_API_KEY')


## 0.5 In-lined helper code


In [ ]:
# Planet quick-search helper.
def GetDataAvailability(apiKey, geom, startDate, endDate):
    # the geo json geometry object we got from geojson.io
    geo_json_geometry = geom

    # filter for items the overlap with our chosen geometry
    geometry_filter = {
    "type": "GeometryFilter",
    "field_name": "geometry",
    "config": geo_json_geometry
    }

    # filter images acquired in a certain date range
    date_range_filter = {
    "type": "DateRangeFilter",
    "field_name": "acquired",
    "config": {
        "gte": startDate + "T00:00:00.000Z",
        "lte": endDate + "T00:00:00.000Z"
    }
    }

    filter = {
    "type": "AndFilter",
    "config": [geometry_filter, date_range_filter]
    }

    search_endpoint_request = {
    "item_types": ["PSScene"],
    "filter": filter
    }

    result = requests.post(
        'https://api.planet.com/data/v1/quick-search',
        auth=HTTPBasicAuth(apiKey, ''),
        json=search_endpoint_request)
    
    requestJson = json.loads(result.text)

    dateTimes = [item['properties']['acquired'] for item in requestJson['features']]

    dates = [item['properties']['acquired'].split('T')[0] for item in requestJson['features']]
    cloudCover = [item['properties']['cloud_cover'] for item in requestJson['features']]
    satelliteID = [item['properties']['satellite_id'] for item in requestJson['features']]
    # eps = [item['properties']['epsg_code'] for item in requestJson['features']]
    geomShape = shape(geom)
    geometries = [shape(item['geometry']) for item in requestJson['features']]
    intersectionGeom = [geomShape.intersection(geom) for geom in geometries]
    intersectionArea = [geom.area / geomShape.area for geom in intersectionGeom]
    df = gpd.GeoDataFrame(geometry=intersectionGeom)
    df['Date'] = dates
    df['DateTime'] = dateTimes
    df['cloudCover'] = cloudCover
    df['IntersectionArea'] = intersectionArea
    df['SatelliteID'] = satelliteID
    df = df.set_crs("EPSG:32646")

    def getWeightedMean(subDf):
        return np.sum(subDf.cloudCover.values * subDf.IntersectionArea.values) / np.sum(subDf.IntersectionArea.values)

    groupedDF0 = pd.DataFrame(df.groupby(['Date','SatelliteID']).apply(lambda x: getWeightedMean(x)))
    groupedDF1 = pd.DataFrame(df.groupby(['Date','SatelliteID']).apply(lambda x: gpd.GeoDataFrame(x['geometry']).dissolve()))

    groupedDF1['IntersectionArea'] = [geomShape.intersection(groupedDF1.iloc[geom_i,:].geometry).area / geomShape.area for geom_i in range(len(groupedDF1))]
    groupedDF1['cloudCover'] = groupedDF0.values

    df = pd.DataFrame(groupedDF1.groupby(['Date']).apply(lambda x: getWeightedMean(x)))

    planetOverlap = pd.DataFrame()
    planetOverlap['Date'] = df.reset_index().Date
    planetOverlap['Overlap'] = [geom.intersection(geomShape).area / geomShape.area for geom in groupedDF1.groupby(['Date']).apply(lambda x: gpd.GeoDataFrame(x['geometry']).dissolve()).geometry]
    planetOverlap['CloudCover'] = df.values

    return planetOverlap


In [ ]:
# Earth Engine collection helpers.
def GetDataFrames(startDate, endDate, geometry, planetAPIKey, planetCSV=None, includeS2TOA=False):
    start_date = ee.Date(startDate)
    end_date = ee.Date(endDate)

    # modis_aqua = ee.ImageCollection("MODIS/061/MYD09GA")
    modis_terra = ee.ImageCollection("MODIS/061/MOD09GA")
    landsat7 = ee.ImageCollection("LANDSAT/LE07/C02/T1")
    landsat8 = ee.ImageCollection("LANDSAT/LC08/C02/T1")
    landsat9 = ee.ImageCollection("LANDSAT/LC09/C02/T1")
    # sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    sentinel2 = ee.ImageCollection("COPERNICUS/S2_SR")
    sentinel2L1 = ee.ImageCollection("COPERNICUS/S2")
    # sentinel2L1 = ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
    sentinel1 = ee.ImageCollection("COPERNICUS/S1_GRD")

    sentinel2_collection = get_imagery_availability(sentinel2, 'S2', start_date, end_date, geometry, 'sentinel2', 20)
    sentinel2L1_collection = get_imagery_availability(sentinel2L1, 'S2 TOA', start_date, end_date, geometry, 'sentinel2_TOA', 20)
    sentinel1_collection = get_imagery_availability(sentinel1, 'S1', start_date, end_date, geometry, None, None)
    # modis_aqua_collection = get_imagery_availability(modis_aqua, 'MODIS', start_date, end_date, geometry, 'modis', 1000)
    modis_terra_collection = get_imagery_availability(modis_terra, 'MODIS', start_date, end_date, geometry, 'modis', 1000)
    landsat7_collection = get_imagery_availability(landsat7, 'L7', start_date, end_date, geometry, 'landsat', 30)
    landsat8_collection = get_imagery_availability(landsat8, 'L8', start_date, end_date, geometry, 'landsat', 30)
    landsat9_collection = get_imagery_availability(landsat9, 'L9', start_date, end_date, geometry, 'landsat', 30)

    date_list = pd.date_range(pd.to_datetime(start_date.getInfo()['value'], unit='ms'), pd.to_datetime(end_date.getInfo()['value'], unit='ms')).tolist()
    date_list_df = pd.DataFrame(data = [date_list], index = ['date']).T
    date_list_df['date'] = date_list_df['date'].dt.date

    sentinel2_df = create_imagery_info_df(sentinel2_collection)
    # If no S2 surface reflectance data, use top of atmosphere S2 data (if available)
    s2TOAFlag=False
    if includeS2TOA==True and len(sentinel2_df)<1:
        sentinel2_df = create_imagery_info_df(sentinel2L1_collection)
        if len(sentinel2_df)>=1:
            s2TOAFlag=True
        else:
            sentinel2_df = create_imagery_info_df(sentinel2_collection)
            s2TOAFlag=False
    sentinel1_df = create_imagery_info_df(sentinel1_collection)
    # modis_aqua_df = create_imagery_info_df(modis_aqua_collection)
    modis_terra_df = create_imagery_info_df(modis_terra_collection)
    landsat7_df = create_imagery_info_df(landsat7_collection)
    landsat8_df = create_imagery_info_df(landsat8_collection)
    landsat9_df = create_imagery_info_df(landsat9_collection)

    if planetCSV==None:
        PlanetInfoDF = GetDataAvailability(planetAPIKey, geometry.getInfo(), startDate, endDate)
        PlanetInfoDF = PlanetInfoDF.rename(columns = {'Date':'date', 'Overlap':'pctArea', 'CloudCover':'pctCloud'})
        PlanetInfoDF = PlanetInfoDF.set_index('date')
        PlanetInfoDF['pctCloud'] = PlanetInfoDF['pctCloud']*100
        PlanetInfoDF['pctArea'] = PlanetInfoDF['pctArea']*100
    elif planetCSV!=None:
        PlanetInfoDF = pd.read_csv(planetCSV)
        PlanetInfoDF = PlanetInfoDF.set_index('date')
        
    return date_list_df, sentinel2_df, sentinel1_df, modis_terra_df, landsat7_df, landsat8_df, landsat9_df, PlanetInfoDF, s2TOAFlag



def bitwiseExtract(value, fromBit, toBit):
    if (toBit == None): 
        toBit = fromBit
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return value.rightShift(fromBit).bitwiseAnd(mask)

# Create function to add cloudy pixels for Sentinel-2 - NOTE: THIS ONE SIMPLY TAKES THE MEAN CLOUD PROBABILITY ACROSS THE IMAGE ROI
def add_clouds_s2(image):
    s2_qa_band = 'MSK_CLDPRB'
    cloudy = image.select(s2_qa_band).divide(100).rename('cloudy') #simply makes copy of the cloud probability layer (scaled to 0 to 1)
    return image.addBands(cloudy)

# Create function to add cloudy pixels for MODIS
def add_clouds_modis(image):
    modis_qa_band = 'state_1km'
    qa = image.select(modis_qa_band)
    clouds = (bitwiseExtract(qa, 0, 1).neq(0).And(bitwiseExtract(qa, 0, 1).neq(3)))
    cirrus = bitwiseExtract(qa, 8, 9).neq(0)
    cloudy = clouds.Or(cirrus).rename('cloudy') # this gives us cloudy pixels
    return image.addBands(cloudy)

# Create function to add cloudy pixels for Landsat
def add_clouds_landsat(image):
    landsat_qa_band = 'QA_PIXEL'
    qa = image.select(landsat_qa_band)
    clouds = bitwiseExtract(qa, 3, None).neq(0)
    dilated_clouds = bitwiseExtract(qa, 1, None).neq(0)
    cloudy = clouds.Or(dilated_clouds).rename('cloudy') # this gives us cloudy pixels
    return image.addBands(cloudy)

def mosaicBy(collection):

    # Return the collection as a list of images
    image_list = ee.ImageCollection(collection).toList(ee.ImageCollection(collection).size())

    # Get all the dates as list
    def get_date(image):
        date = ee.Image(image).date().format("YYYY-MM-dd")
        return date

    all_dates = image_list.map(get_date)

    # Get distinct dates
    dates_unique = all_dates.distinct()

    # Create mosaic of images on the same day
    def mosaic_day(day):

        # Split into components
        day1 = ee.String(day).split(" ")
        date1 = ee.Date(day1.get(0))

        # Filter with start date and date + 1 day, to an imageCollection
        imagecol = ee.ImageCollection(collection).filterDate(date1, date1.advance(1, "day"))

        # Mosaic the daily imageCollection - *** should this be specified to maximise the cloud-free pixels?
        image = imagecol.mosaic()

        # # Do a quality mosaic of the daily imageCollection
        # def quality_mosaic_prep(image):
        #   cldProb = image.select(sensor_mosaic_band)
        #   cldProbInv = cldProb.multiply(-1).rename('quality')
        #   return image.addBands(cldProbInv)
        #
        # imagecol = imagecol.map(quality_mosaic_prep)
        # image = imagecol.qualityMosaic(quality)

        # Perform a dissolve of the daily imageCollection geometries to get the mosaic image extent
        mosaic_geom = imagecol.geometry().dissolve()

        return ee.Image(image).set(
            "system:time_start", date1.millis(),
            "system:date", date1.format("YYYY-MM-dd"),
            "system:id", day1,
            'system:footprint', mosaic_geom); # preserve the geometry

    mosaic_image_list = ee.ImageCollection(dates_unique.map(mosaic_day))

    return mosaic_image_list

# 3) Create a function which combines functions to get info about the available imagery
def get_imagery_availability(collection, collectionName, start, end, roi, sensor_cloud_flag, sensor_cloud_scale):
    
    # 3.0) Filter the ImageCollection for the ROI and TOI
    filteredCollection = collection.filterDate(ee.Date(start), ee.Date(end)).filter(ee.Filter.bounds(roi))
    n = 0
    try:
        n = filteredCollection.size().getInfo()
    except:
        n = 0
    
    if n == 0: # stop the function if no images for the ROI and TOI
        print(collectionName, ': No images for the TOI and ROI')
        return
    
    # 3.1) Add cloud pixels to the image, if MODIS, S2 or Landsat
    if sensor_cloud_flag == 'modis':
        filteredCollection_wClouds = filteredCollection.map(add_clouds_modis)
    elif sensor_cloud_flag == 'sentinel2':
        filteredCollection_wClouds = filteredCollection.map(add_clouds_s2)
    elif sensor_cloud_flag == 'landsat':
        filteredCollection_wClouds = filteredCollection.map(add_clouds_landsat)
    else:
        filteredCollection_wClouds = filteredCollection
    
    
    # 3.2) Apply the mosaic function on the filteredCollection to mosaic images by day
    mosaicCollection = mosaicBy(filteredCollection_wClouds)
    
    
    # Convert the ROI to geometry and get its area, for performing the clip and intersection
    roi_geom = roi
    roi_area = roi_geom.area()
    
    # 3.3) Create a function to clip the image to the ROI, map to the collection
    def clip_to_roi(image):
        return image.clip(roi)

    
    # 3.3) Apply the clipping function on the fiteredCollection mosaic - NOT PERFORMED
    #mosaicCollectionClipped = mosaicCollection.map(clip_to_roi)
    mosaicCollectionClipped = mosaicCollection
    
   
    # 3.4) Create a function to compute the % of ROI area covered by the image, set as a new property
    def get_pct_area_intersect(image):
        image_geom = image.geometry()
        intersect = roi_geom.intersection(image_geom) #roi_geom.intersection(bbox, 1)
        intersect_area = intersect.area()#.getInfo()
        pct_area = intersect_area.divide(roi_area).multiply(100)#.getInfo()
        return image.set('PctArea', pct_area)
    
    # Apply the get area intersect function on the collection
    mosaicCollectionClipped_wArea = mosaicCollectionClipped.map(get_pct_area_intersect)

    
    # 3.5) Create a function to get the cloudiness of the image  and set it as a new property, only if optical imagery
    # Create function to reduce cloud values must be run inside the main function as uses roi and sensor_cloud_scale
    def reduce_clouds(image):
        pctCloud = (image.select('cloudy').reduceRegion(reducer = ee.Reducer.mean(), # taking the mean is same as taking the sum divided by the total pixels
                                                            geometry = roi_geom.intersection(image.geometry()), #roi_geom,
                                                            scale = sensor_cloud_scale,
                                                            bestEffort = True)) # sensor_cloud_scale: MODIS cloud mask resolution is 1 km // S2 is 60 m // Landsat is 30 m
        pctCloudVal = ee.Number(pctCloud.get('cloudy')).multiply(100)
        return image.set('PctCloud', pctCloudVal) # Add as a property
    
    def reduce_clouds_dummy(image):
        return image.set('PctCloud', 0) # Add as a property
    
    # Apply the reduce clouds function to get a single cloudiness value for the ROI
    if sensor_cloud_flag in ['modis', 'sentinel2', 'landsat']:
        mosaicCollectionClipped_wArea_wCloudInfo = mosaicCollectionClipped_wArea.map(reduce_clouds)
    else:
        mosaicCollectionClipped_wArea_wCloudInfo = mosaicCollectionClipped_wArea.map(reduce_clouds_dummy)
     
    # # Create a function to extract cloudiness data from a MODIS image
    # def get_cloudiness_modis(image):
    #   # Cloud state is in the low two bits of the 'state_1km' band
    #   cloudState = image.select('state_1km').bitwiseAnd(0x03)
    #   # 0 means clear, 1 means cloudy, 2 means mixed, and 3 means missing data, convert to the range 0.0-1.0 and mask out the missing pixels
    #   return cloudState.float().where(cloudState.eq(2), 0.5).updateMask(cloudState.neq(3))

    # Return the imageCollection, filtered to the ROI and TOI, mosaicded, with the useful info as new properties
    print(collectionName, ': Success in getting the FilteredCollection with image info')
    return mosaicCollectionClipped_wArea_wCloudInfo

# Create a function to extract the dates, % area of ROI and % cloudiness into a data frame
# And then convert to a time series of all dates between the start and end date, filled in with the relevant info where images are available and -1 where not
def create_imagery_info_df(collection):
    
    if collection == None:
        return pd.DataFrame()
    
    # Extract the info as separate numpy arrays
    dateTime = np.array(collection.aggregate_array('system:time_start').getInfo())
    pctArea = np.array(collection.aggregate_array('PctArea').getInfo())
    try:
        pctCloud = np.array(collection.aggregate_array('PctCloud').getInfo())
    except:
        pctCloud = np.ones(pctArea.shape) * 50.0
        
    # Create a pandas dataframe combining the arrays
    collection_df = pd.DataFrame(data = [dateTime, pctArea, pctCloud], index = ['dateTime', 'pctArea', 'pctCloud']).T
    
    # Convert the date format and add a simplified date column
    collection_df['dateTime'] = pd.to_datetime(collection_df['dateTime'], unit='ms')
    collection_df['date'] = collection_df['dateTime'].dt.date
    
    # Set the date as the index
    collection_df = collection_df.set_index('date')
    
    # # Merge the collection_info_df with the date_list to update values where there was available imagery
    # collection_df_full = pd.merge(date_list_df, collection_df, on='date', how='outer').set_index('date')

    # Return both the original and the full (time series) dataframes of info about the imagery
    return collection_df


In [ ]:
# Plotting helpers.
def visualize_imagery_info(collection_df, ax, j):
    
    if collection_df.empty:
        return

    # Define x and y values for plotting
    x = collection_df.index
    y = [j] * len(collection_df)
    
    # # Use the cloudiness values as the % opacity of the marker (alpha) 
    # alphas = np.array((collection_df_full['pctCloud'].fillna(0))/100)
    
    # Use the cloudiness value as the color of the marker, from white (no clouds) to grey (clouds)
    fills = np.array(collection_df['pctCloud']/100)

    # Use the % area coverage as the internal marker size, inside a 100% outer marker
    mdim = 30 # marker dimension
    msize = mdim**2 # marker size (square of dimension, since using area of a square)
    sizes = np.array((mdim*(collection_df['pctArea'])/100)**2)
    
    # Plot as scatter
    ax = ax or plt.gca()
    ax.scatter(x, y, alpha=1, facecolors='none', edgecolors='gray', marker='s', linestyle='-', s=msize, hatch='///') # outer edge for theoretical full 100% area
    ax.scatter(x, y, alpha=1, marker='s', edgecolors='none', c=fills, cmap='gray', vmin=0, vmax=1, s=sizes) # fill for cloudiness
    ax.scatter(x, y, alpha=1, facecolors='none', edgecolors='gray', marker='s', linestyle='-', s=sizes) # inner edge for actual % area coverage
    #ax.scatter(x, y, alpha=alphas, marker='o', edgecolors='none', facecolors='gray', s=sizes) # fill for cloudiness
    
    return ax

# Function to create the gra[hoc
def showImageAvailability(startDate, endDate, geometry, planetAPIKey, planetCSV=None, includeS2TOA=False):
    
    # Run GetDataFrames function to get dataframes for each sat-sensor
    date_list_df, sentinel2_df, sentinel1_df, modis_terra_df, landsat7_df, landsat8_df, landsat9_df, PlanetInfoDF, s2TOAFlag = GetDataFrames(startDate, endDate, geometry, planetAPIKey, planetCSV, includeS2TOA)

    # Create a second plot where the sat image availability is grouped by sat / sensor type
    # Define names for plotting
    if s2TOAFlag==True:
        names = ['Sentinel-1', 'Sentinel-2 TOA', 'MODIS', 'Landsat', 'Planet']
    else:
        names = ['Sentinel-1', 'Sentinel-2', 'MODIS', 'Landsat', 'Planet']
    js = range(len(names), 0, -1)

    # Create the plot
    fig, ax = plt.subplots(figsize=(len(date_list_df)/1.5, len(names)+1)) #0.75 * len(names)+1
    bottom, top = 0, 0.88
    left, right = 0, 1
    fig.subplots_adjust(top=top, bottom=bottom, left=left, right=right, hspace=0.1, wspace=0.1)

    # Run the function to add each imagery info to the plot - ***can remove or add to this (i.e., plot more or fewer sats/sensors), so long as adjust the list 'names' and adjust the js[n] in the below functions
    visualize_imagery_info(sentinel1_df, ax, js[0])
    visualize_imagery_info(sentinel2_df, ax, js[1])
    visualize_imagery_info(modis_terra_df, ax, js[2]) # visualize_imagery_info(modis_aqua_df, ax, js[3])
    visualize_imagery_info(landsat7_df, ax, js[3])
    visualize_imagery_info(landsat8_df, ax, js[3])
    visualize_imagery_info(landsat9_df, ax, js[3])
    visualize_imagery_info(PlanetInfoDF, ax, js[4])

    # Format the ticks and gridlines
    major_tick = MultipleLocator(7)
    minor_tick = MultipleLocator(1)
    ax.xaxis.set_major_locator(major_tick)
    ax.xaxis.set_minor_locator(minor_tick)
    ax.tick_params(axis='x', which='major', labelsize=22)
    plt.yticks(ticks= range(len(names), 0, -1), labels=names, fontsize=22)
    ax.grid(False, axis = 'x')
    #ax.grid(True, axis = 'x', which='minor', linestyle=':')
    ax.grid(False, axis = 'y')


    # Format the plot title
    # fig.suptitle('Image availability for ' + nameArg, fontsize=24, x=0.15, y=0.88)

    # Format the x and y limits
    plt.xlim(date_list_df['date'].min()-timedelta(days=1), date_list_df['date'].max())
    plt.ylim(0.5, len(names)+0.5)

    # Add a marker legend for % area coverage
    mdim = 30 # marker dimension
    square = mlines.Line2D([], [], color='black', marker='s', linestyle='none', fillstyle='none', #marker='$\u2B1A$'
                            markersize=mdim, label='Image % of ROI coverage')
    square2 = mlines.Line2D([], [], color='black', marker='s', linestyle='none', fillstyle='none',
                            markersize=mdim/2, label='')
    plt.legend(handles=[square, square2], bbox_to_anchor=(0.50, 1.18), borderaxespad=0.1, frameon=False, labelspacing = -1.0, fontsize=16)

    # Add a cbar legend for % cloudiness
    #cbar_ax = fig.add_axes([0.88, 0.94, 0.08, 0.04]) # currently need to be done manually
    # cbar_ax = fig.add_axes([0.52, 0.96, 0.05, 0.04]) # currently need to be done manually
    cbar_ax = fig.add_axes([0.62, 0.88, 0.05, 0.04]) # currently need to be done manually
    cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=100), cmap='gray'), orientation='horizontal', cax=cbar_ax)
    cbar.set_ticks([0,100])
    cbar.set_ticklabels([0,100])
    cbar.ax.tick_params(labelsize=12)
    cbar.set_label(label='Image % cloudiness', size=16, x = 2.6, labelpad=-33)

    fig.tight_layout()

    # Finalise the plot
    plt.show()
    return fig


## 1. Define region and time of interest

### Region of Interest
This is easiest to do by calling a FeatureCollection of an asset (i.e. shapefile or geojson) already uploaded to GEE.

In [ ]:
# Define the region of interest - using shapefile for Rio_Grande
roi = ee.FeatureCollection("users/alex-saunders/houston") # will need to upload shapefile to assets in your own GEE account, or we switch this for shapefile to reside in a earth engine cloud project
# Get number of elements in the region of interest file (e.g., could have multiple watersheds / admin areas in the same shapefile)
n = roi.size().getInfo()

__Choose to get image availability for each geometry within the feature collection or dissolve the geometries and get a single image availability__

If you want to retain all original geometries, do not run the below

In [ ]:
# Dissolve to one element
roi = ee.FeatureCollection(roi.geometry().dissolve())
n = roi.size().getInfo()

### Time of interest
__Guidance:__
* Currently define the "central date" you want to look for and the number of days either side.
* If preferred can manually specify start and end date as strings in correct format - see "options to edit in below code"
* Can provide multiple dates and multiple images will be created.

In [ ]:
# Define the dates of interest and the number of days either side of the target date for whcih to create the visualization
# Add multiple dates to search for multiple time periods / events at once
dates = ['8/27/2017', '8/15/2021'] 
dates = [datetime.datetime.strptime(item,'%m/%d/%Y') for item in dates]
daysBeforeAfter = 12 # days

### Optional: manually upload a CSV with cloud cover for Planet images, extracted manually from Planet Explorer
Since we know that currently, the Planet data does not return properly using the API (for given dates?), it is sensible to manually create a CSV file of Planet images with their cloud cover and % area coverage from Planet explorer.

In [ ]:
# Use the Planet API by default. Set this to a local CSV path to override.
planetCSV = None


## 2. Create graphic of image availability

__Planet API key is loaded from `/home/cefect/.config/USFloods_inference/secrets.toml`__


In [ ]:
# Planet API key is loaded above from the local secrets file.
assert planetKey


### User options to edit in below code:
* __Replace planetCSV=None__ with location of Planet csv file if want to include it, otherwise the default Planet API will be used-this can give erroneous results
* __Replace includeS2TOA=False__ with includeS2TOA=True if want to include Sentinel-2 top of atmosphere data if there is no surface reflectance data available - S2 SR data tends to start around end of 2018, whereas TOA data is available from roughly 2015-16 onwards
* __Replace startDateStr and endDateStr__ if you want to manually define start and end dates, rather than rely on the centre dates and buffer days created above

In [ ]:
# Loop through event dates - will produce as many images as there are event dates
for date in dates:
    
    # Loop through elements of the ROI (if more than one, otherwise will just create a single output for the single ROI)
    for k in range(n): 
        
        # Get geometry as geometry format, from the Feature Collection
        geometry = ee.Feature(roi.toList(n).get(k)).geometry() 
        
        # Create start and end date in required format for GEE search
        startDateStr = (date - timedelta(days=daysBeforeAfter)).strftime('%Y-%m-%d')
        endDateStr = (date + timedelta(days=daysBeforeAfter)).strftime('%Y-%m-%d')
        
        # If you want to use manually defined start and end rather than a central date with a days before / after, specify as string instead
        # startDateStr = '2017-08-01'
        # endDateStr = '2017-09-01'
        
        # Create figure
        # fig = showImageAvailability(startDateStr, endDateStr, geometry, planetKey, planetCSV=None, includeS2TOA=False) 
        fig = showImageAvailability(startDateStr, endDateStr, geometry, planetKey, planetCSV=planetCSV, includeS2TOA=True) 
        
        # Save figure to png
        fig.savefig('imageAvailabilityFigure.png')


### NOTE
If you want to get a dataframe of the actual data that creates the graphic, you can run the below

In [ ]:
# Get dataframes of info
date_list_df, sentinel2_df, sentinel1_df, modis_terra_df, landsat7_df, landsat8_df, landsat9_df, PlanetInfoDF, s2TOAFlag = GetDataFrames(startDateStr, endDateStr, geometry, planetKey, planetCSV=planetCSV, includeS2TOA=False)


In [ ]:
sentinel1_df

In [ ]:
PlanetInfoDF